# Stage 2 — Evaluate the SFT model

Evaluates `./vi-smollm-135m-sft/` on ChatML-formatted prompts.

**Prerequisites:** Stage 1 (pretrain) + Stage 2 (SFT) done — `uv run python finetuning/train.py --mode sft`

**Checks:**
1. Chat format: the model answers and **stops at `<|im_end|>`** instead of running on
2. Response quality (qualitative, greedy + sampled)
3. Degeneration: repetition rate and stop rate
4. Comparison with the Stage 1 base model on the same prompts
5. Training curve

> Note: `generate_sample` stops at `<|im_end|>` explicitly. The SFT model's saved `generation_config.json`
> (from before the fix in `run_sft`) lists only `</s>`, so plain `model.generate()` keeps producing new `assistant` turns.

In [1]:
import os
import sys

def _find_repo_root():
    d = os.getcwd()
    while not os.path.isdir(os.path.join(d, "finetuning")):
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError("Run this notebook from inside the ViRL-SmolLM2-135M repo")
        d = parent
    return d

REPO_ROOT = _find_repo_root()
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

In [2]:
import json
import torch
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast

from utils.eval_utils import generate_sample

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

MODEL_DIR = "./vi-smollm-135m-sft"
BASE_DIR = "./vi-smollm-135m-pretrain"
assert os.path.isdir(MODEL_DIR), (
    f"{MODEL_DIR} not found — run Stage 2 first:\n"
    "  uv run python finetuning/train.py --mode sft"
)

def load(path):
    return AutoModelForCausalLM.from_pretrained(path, dtype=DTYPE).to(DEVICE).eval()

model = load(MODEL_DIR)
tokenizer = PreTrainedTokenizerFast.from_pretrained("./vi_smollm_tokenizer")
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅ SFT model loaded from {MODEL_DIR} ({n_params:.1f}M params) on {DEVICE}")
print("saved eos_token_id:", model.generation_config.eos_token_id, "| <|im_end|> id:", tokenizer.convert_tokens_to_ids("<|im_end|>"))

/home/thuandn/Repository/ViRL-SmolLM2-135M/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 2558.04it/s]


✅ SFT model loaded from ./vi-smollm-135m-sft (124.6M params) on cuda
saved eos_token_id: [2, 0] | <|im_end|> id: 6


## 1–2. Chat format and responses

Greedy with a mild repetition penalty, plus one sample at temperature 0.5. `max_new_tokens=150`; `stopped` = ended on `<|im_end|>`/`</s>` before the limit.

In [3]:
SYSTEM = "You are a helpful Vietnamese assistant."
test_prompts = [
    "Mặt trời mọc ở đâu?",
    "Việt Nam thủ đô là thành phố nào?",
    "Cho tôi một công thức nấu phở bò.",
    "Xin chào, bạn là ai?",
    "Hãy giải thích trí tuệ nhân tạo là gì trong hai câu.",
    "Viết một câu chào buổi sáng.",
]
MAX_NEW = 150

def chat_prompt(q):
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}],
        tokenize=False, add_generation_prompt=True)

def n_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

GREEDY = dict(do_sample=False, repetition_penalty=1.15)
SAMPLE = dict(do_sample=True, temperature=0.5, top_p=0.9, repetition_penalty=1.15)

def run(m, prompts, **gen):
    torch.manual_seed(0)
    outs = []
    for q in prompts:
        a = generate_sample(m, tokenizer, chat_prompt(q), max_new_tokens=MAX_NEW, **gen)
        outs.append(a)
    return outs

greedy_out = run(model, test_prompts, **GREEDY)
sample_out = run(model, test_prompts, **SAMPLE)
for q, g, s in zip(test_prompts, greedy_out, sample_out):
    print(f"Q: {q}\n  greedy [{n_tokens(g)} tok]: {g}\n  sample [{n_tokens(s)} tok]: {s}\n")

assert all(r.strip() for r in greedy_out + sample_out), "Some generations were empty"

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q: Mặt trời mọc ở đâu?
  greedy [39 tok]: Để tạo ra một bức tranh thiên nhiên đẹp, bạn có thể sử dụng các công cụ như: máy ảnh kỹ thuật số và máy quay phim để làm cho nó trở nên sống động hơn.
  sample [150 tok]: Từ khóa: mặt trời, thành phố, hoa, nắng, gió, mưa, mùa hè, sông, biển, bờ biển, mây, biển, thời tiết, sương mù, hoa, cỏ cây, đất, ao, hồ, nước, sông, biển, biển, phía Nam, biển, vịnh, vùng, biển, miền trung, biển, đảo, khu vực, biển, núi, biển, biển, biển, mặt trời, biển, biển, trong đó, mặt trời, biển, biển, biển, biển, bờ, biển, biển, miền trung, biển, biển, biển, biển, biển, biển, biển, biển, biển, từ, phần, đường, biển, bờ, biển, biển, biển

Q: Việt Nam thủ đô là thành phố nào?
  greedy [149 tok]: Thành phố này có một địa điểm du lịch nổi tiếng, đó chính là Thành Phố Hồ Chí Minh. Đây là nơi có nhiều danh lam thắng cảnh đẹp và những công trình kiến trúc độc đáo.

Địa chỉ: Số 2 đường Nguyễn Văn Cừ, phường Tân Phú, quận 7, TP HCM.

Tên gọi khác: Thành Phố Hồ Chí Minh (tên gọi

## 3. Degeneration metrics

- **stop rate**: fraction of answers that end before `max_new_tokens` (a model that learned the format stops on its own)
- **echo rate**: answer repeats the question or contains role markers (`assistant`, `user`)
- **3-gram repetition**: fraction of repeated word 3-grams (lower is better)

In [4]:
def rep_ngram_rate(text, n=3):
    w = text.split()
    grams = [tuple(w[i:i+n]) for i in range(len(w) - n + 1)]
    return 0.0 if not grams else 1 - len(set(grams)) / len(grams)

def metrics(prompts, outs):
    stop = sum(n_tokens(o) < MAX_NEW - 2 for o in outs) / len(outs)
    echo = sum(("assistant\n" in o) or (q in o) for q, o in zip(prompts, outs)) / len(outs)
    rep = sum(rep_ngram_rate(o) for o in outs) / len(outs)
    avg = sum(n_tokens(o) for o in outs) / len(outs)
    return {"stop_rate": stop, "echo_rate": echo, "rep3": rep, "avg_tokens": avg}

print("SFT greedy :", {k: round(v, 3) for k, v in metrics(test_prompts, greedy_out).items()})
print("SFT sample :", {k: round(v, 3) for k, v in metrics(test_prompts, sample_out).items()})

SFT greedy : {'stop_rate': 0.667, 'echo_rate': 0.0, 'rep3': 0.013, 'avg_tokens': 75.833}
SFT sample : {'stop_rate': 0.667, 'echo_rate': 0.0, 'rep3': 0.031, 'avg_tokens': 76.5}


## 3b. Larger evaluation (30 prompts)

6 prompts are too few: one answer moves a rate by 17 points. This set has 30 prompts in 5 categories.

- **stopped**: the last generated token is `<|im_end|>`/`</s>` (exact, not inferred from length)
- **keyword hit** (facts only): the answer contains an expected keyword — a crude correctness check
- **Vietnamese**: share of letters that are Vietnamese-specific (diacritics) or plain Latin; flags code-switching/garbage
- 95% Wilson intervals show how little 30 prompts can tell you

In [ ]:
import math

EVAL_SET = {
    "fact": [  # (prompt, accepted keywords)
        ("Thủ đô của Việt Nam là gì?", ["Hà Nội"]),
        ("Thành phố nào đông dân nhất Việt Nam?", ["Hồ Chí Minh", "Sài Gòn"]),
        ("Việt Nam nằm ở khu vực nào của châu Á?", ["Đông Nam Á"]),
        ("Sông nào dài nhất Việt Nam?", ["Mê Kông", "Mekong", "Cửu Long"]),
        ("Tết Nguyên Đán diễn ra vào mùa nào?", ["xuân", "mùa xuân"]),
        ("Quốc kỳ Việt Nam có màu gì?", ["đỏ"]),
        ("Ai là chủ tịch đầu tiên của nước Việt Nam Dân chủ Cộng hòa?", ["Hồ Chí Minh"]),
        ("Một năm có bao nhiêu tháng?", ["12", "mười hai"]),
        ("Nước nào có thủ đô là Tokyo?", ["Nhật"]),
        ("Mặt trời mọc ở hướng nào?", ["đông"]),
    ],
    "howto": [
        "Cho tôi một công thức nấu phở bò.",
        "Làm thế nào để học tiếng Anh hiệu quả?",
        "Hướng dẫn cách pha một tách cà phê sữa đá.",
        "Làm sao để ngủ ngon hơn?",
        "Cách giữ gìn sức khỏe mùa đông là gì?",
        "Làm thế nào để tiết kiệm tiền hàng tháng?",
    ],
    "explain": [
        "Hãy giải thích trí tuệ nhân tạo là gì trong hai câu.",
        "Biến đổi khí hậu là gì?",
        "Giải thích ngắn gọn về quang hợp.",
        "Internet hoạt động như thế nào?",
        "Tại sao bầu trời có màu xanh?",
    ],
    "write": [
        "Viết một câu chào buổi sáng.",
        "Viết một đoạn ngắn giới thiệu về Hà Nội.",
        "Viết lời cảm ơn gửi tới thầy cô giáo.",
        "Hãy viết một bài thơ ngắn về mùa thu.",
    ],
    "chat": [
        "Xin chào, bạn là ai?",
        "Bạn có thể giúp tôi điều gì?",
        "Hôm nay bạn thế nào?",
        "Cảm ơn bạn nhé!",
        "Tạm biệt!",
    ],
}
EVAL_PROMPTS = [(c, p if isinstance(p, str) else p[0], [] if isinstance(p, str) else p[1])
                for c, ps in EVAL_SET.items() for p in ps]
print(len(EVAL_PROMPTS), "prompts:", {c: len(v) for c, v in EVAL_SET.items()})

STOP_IDS = {tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")}

@torch.no_grad()
def gen(m, question, max_new=MAX_NEW, **kw):
    """Return (text, stopped). `stopped` = last new token is a stop token."""
    x = tokenizer(chat_prompt(question), return_tensors="pt").to(DEVICE)
    x.pop("token_type_ids", None)
    out = m.generate(**x, max_new_tokens=max_new, eos_token_id=sorted(STOP_IDS),
                     pad_token_id=tokenizer.pad_token_id, **kw)[0][x["input_ids"].shape[1]:]
    stopped = out[-1].item() in STOP_IDS
    return tokenizer.decode(out, skip_special_tokens=True).strip(), stopped

def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0, c - h), min(1, c + h))

def vi_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters: return 0.0
    ok = sum(ch.isascii() or ch.lower() in "ăâđêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ" for ch in letters)
    return ok / len(letters)

def evaluate(m, **kw):
    torch.manual_seed(0)
    rows = []
    for cat, q, kws in EVAL_PROMPTS:
        a, stopped = gen(m, q, **kw)
        rows.append({"cat": cat, "q": q, "a": a, "stopped": stopped,
                     "echo": ("assistant\n" in a) or (q in a),
                     "rep3": rep_ngram_rate(a), "vi": vi_ratio(a),
                     "hit": (any(k.lower() in a.lower() for k in kws) if kws else None)})
    return pd.DataFrame(rows)

def summarize(df):
    n = len(df)
    st = df.stopped.sum(); lo, hi = wilson(st, n)
    f = df[df.hit.notna()]; fk = int(f.hit.sum()); flo, fhi = wilson(fk, len(f))
    return {"stopped": f"{st}/{n} ({st/n:.0%}, CI {lo:.0%}-{hi:.0%})",
            "echo": f"{df.echo.mean():.0%}", "rep3": round(df.rep3.mean(), 3),
            "vietnamese": round(df.vi.mean(), 3),
            "fact keyword hit": f"{fk}/{len(f)} ({fk/len(f):.0%}, CI {flo:.0%}-{fhi:.0%})"}

import pandas as pd
sft_df = evaluate(model, **GREEDY)
for k, v in summarize(sft_df).items(): print(f"{k:>18}: {v}")
print("\nPer category (stopped rate / rep3):")
print(sft_df.groupby("cat").agg(stopped=("stopped", "mean"), rep3=("rep3", "mean"), n=("q", "count")).round(2))

Fact answers (the part a user would notice first):

In [ ]:
for _, r in sft_df[sft_df.cat == "fact"].iterrows():
    print(f"{'✅' if r.hit else '❌'} {r.q}\n     {r.a[:160]!r}")

## 3b. Larger evaluation (30 prompts)

6 prompts are too few: one answer moves a rate by 17 points. This set has 30 prompts in 5 categories.

- **stopped**: the last generated token is `<|im_end|>`/`</s>` (exact, not inferred from length)
- **keyword hit** (facts only): the answer contains an expected keyword — a crude correctness check
- **Vietnamese**: share of letters that are Vietnamese-specific (diacritics) or plain Latin; flags code-switching/garbage
- 95% Wilson intervals show how little 30 prompts can tell you

In [ ]:
import math

EVAL_SET = {
    "fact": [  # (prompt, accepted keywords)
        ("Thủ đô của Việt Nam là gì?", ["Hà Nội"]),
        ("Thành phố nào đông dân nhất Việt Nam?", ["Hồ Chí Minh", "Sài Gòn"]),
        ("Việt Nam nằm ở khu vực nào của châu Á?", ["Đông Nam Á"]),
        ("Sông nào dài nhất Việt Nam?", ["Mê Kông", "Mekong", "Cửu Long"]),
        ("Tết Nguyên Đán diễn ra vào mùa nào?", ["xuân", "mùa xuân"]),
        ("Quốc kỳ Việt Nam có màu gì?", ["đỏ"]),
        ("Ai là chủ tịch đầu tiên của nước Việt Nam Dân chủ Cộng hòa?", ["Hồ Chí Minh"]),
        ("Một năm có bao nhiêu tháng?", ["12", "mười hai"]),
        ("Nước nào có thủ đô là Tokyo?", ["Nhật"]),
        ("Mặt trời mọc ở hướng nào?", ["đông"]),
    ],
    "howto": [
        "Cho tôi một công thức nấu phở bò.",
        "Làm thế nào để học tiếng Anh hiệu quả?",
        "Hướng dẫn cách pha một tách cà phê sữa đá.",
        "Làm sao để ngủ ngon hơn?",
        "Cách giữ gìn sức khỏe mùa đông là gì?",
        "Làm thế nào để tiết kiệm tiền hàng tháng?",
    ],
    "explain": [
        "Hãy giải thích trí tuệ nhân tạo là gì trong hai câu.",
        "Biến đổi khí hậu là gì?",
        "Giải thích ngắn gọn về quang hợp.",
        "Internet hoạt động như thế nào?",
        "Tại sao bầu trời có màu xanh?",
    ],
    "write": [
        "Viết một câu chào buổi sáng.",
        "Viết một đoạn ngắn giới thiệu về Hà Nội.",
        "Viết lời cảm ơn gửi tới thầy cô giáo.",
        "Hãy viết một bài thơ ngắn về mùa thu.",
    ],
    "chat": [
        "Xin chào, bạn là ai?",
        "Bạn có thể giúp tôi điều gì?",
        "Hôm nay bạn thế nào?",
        "Cảm ơn bạn nhé!",
        "Tạm biệt!",
    ],
}
EVAL_PROMPTS = [(c, p if isinstance(p, str) else p[0], [] if isinstance(p, str) else p[1])
                for c, ps in EVAL_SET.items() for p in ps]
print(len(EVAL_PROMPTS), "prompts:", {c: len(v) for c, v in EVAL_SET.items()})

STOP_IDS = {tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")}

@torch.no_grad()
def gen(m, question, max_new=MAX_NEW, **kw):
    """Return (text, stopped). `stopped` = last new token is a stop token."""
    x = tokenizer(chat_prompt(question), return_tensors="pt").to(DEVICE)
    x.pop("token_type_ids", None)
    out = m.generate(**x, max_new_tokens=max_new, eos_token_id=sorted(STOP_IDS),
                     pad_token_id=tokenizer.pad_token_id, **kw)[0][x["input_ids"].shape[1]:]
    stopped = out[-1].item() in STOP_IDS
    return tokenizer.decode(out, skip_special_tokens=True).strip(), stopped

def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0, c - h), min(1, c + h))

def vi_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters: return 0.0
    ok = sum(ch.isascii() or ch.lower() in "ăâđêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ" for ch in letters)
    return ok / len(letters)

def evaluate(m, **kw):
    torch.manual_seed(0)
    rows = []
    for cat, q, kws in EVAL_PROMPTS:
        a, stopped = gen(m, q, **kw)
        rows.append({"cat": cat, "q": q, "a": a, "stopped": stopped,
                     "echo": ("assistant\n" in a) or (q in a),
                     "rep3": rep_ngram_rate(a), "vi": vi_ratio(a),
                     "hit": (any(k.lower() in a.lower() for k in kws) if kws else None)})
    return pd.DataFrame(rows)

def summarize(df):
    n = len(df)
    st = df.stopped.sum(); lo, hi = wilson(st, n)
    f = df[df.hit.notna()]; fk = int(f.hit.sum()); flo, fhi = wilson(fk, len(f))
    return {"stopped": f"{st}/{n} ({st/n:.0%}, CI {lo:.0%}-{hi:.0%})",
            "echo": f"{df.echo.mean():.0%}", "rep3": round(df.rep3.mean(), 3),
            "vietnamese": round(df.vi.mean(), 3),
            "fact keyword hit": f"{fk}/{len(f)} ({fk/len(f):.0%}, CI {flo:.0%}-{fhi:.0%})"}

import pandas as pd
sft_df = evaluate(model, **GREEDY)
for k, v in summarize(sft_df).items(): print(f"{k:>18}: {v}")
print("\nPer category (stopped rate / rep3):")
print(sft_df.groupby("cat").agg(stopped=("stopped", "mean"), rep3=("rep3", "mean"), n=("q", "count")).round(2))

Fact answers (the part a user would notice first):

In [ ]:
for _, r in sft_df[sft_df.cat == "fact"].iterrows():
    print(f"{'✅' if r.hit else '❌'} {r.q}\n     {r.a[:160]!r}")

## 4. SFT vs Stage 1 base model

Same 30 ChatML prompts through the base model: it has never seen the template, so it should fail to stop and to answer — the gap is what SFT added.

In [5]:
base = load(BASE_DIR)
base_df = evaluate(base, **GREEDY)
for q, a in list(zip(base_df.q, base_df.a))[:2]:
    print(f"Q: {q}\n  BASE : {a[:300]!r}\n")
del base
if DEVICE == "cuda":
    torch.cuda.empty_cache()

pd.DataFrame({"base (stage 1)": summarize(base_df), "sft (stage 2)": summarize(sft_df)}).T

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 2960.27it/s]


Q: Mặt trời mọc ở đâu?
  BASE : 'Trang chủ » Tin tức » Nỗ lực của các nhà lãnh đạo trong cuộc chiến chống lại sự xâm lược của Trung Quốc và những người ủng hộ chính phủ Mỹ là gì?\nPosted on Tháng Mười Một 30, 2021 by admin\nCuộc chiến chống lại sự xâm lược của Trung Quốc và những người ủng hộ chính phủ Mỹ là gì?\nTrong một bài viết mớ'

Q: Việt Nam thủ đô là thành phố nào?
  BASE : 'Home » Việt Nam thủ đô là thành phố nào? DNNN\nPosted on Tháng Mười Hai 1, 2022 by admin\nTrong những năm gần đây, các doanh nghiệp đã và đang có xu hướng phát triển mạnh mẽ hơn trong lĩnh vực kinh tế. Tuy nhiên, việc thực hiện chính sách thuế đối với hoạt động sản xuất kinh doanh của doanh nghiệp vẫn'



,model,stop_rate,echo_rate,rep3,avg_tokens
0,base (stage 1),0.000,0.167,0.195,150.000
1,sft (stage 2),0.667,0.000,0.013,75.833


## 5. Training curve

In [6]:
import glob, re
ckpts = sorted(glob.glob(f"{MODEL_DIR}/checkpoint-*"), key=lambda p: int(re.search(r"checkpoint-(\d+)", p).group(1)))
state = json.load(open(f"{ckpts[-1]}/trainer_state.json"))
hist = [h for h in state["log_history"] if "loss" in h]
n = max(1, len(hist) // 8)
print(f"{len(hist)} log points from {os.path.basename(ckpts[-1])}")
print(f"{'step':>6} {'loss':>7} {'token_acc':>10}")
for h in hist[::n] + [hist[-1]]:
    print(f"{h['step']:>6} {h['loss']:>7.3f} {h.get('mean_token_accuracy', float('nan')):>10.3f}")

1875 log points from checkpoint-18753
  step    loss  token_acc
    10   3.466      0.442
  2350   2.221      0.564
  4690   2.224      0.575
  7030   2.109      0.578
  9370   2.140      0.587
 11710   2.111      0.583
 14050   2.085      0.595
 16390   2.229      0.572
 18730   2.180      0.577
 18750   2.159      0.571


## Reading the results

- A 135M base model with held-out PPL ≈ 55 will produce fluent but often **wrong** answers; SFT teaches the *format* (answer, then stop), not facts.
- Healthy signs: high stop rate, near-zero echo rate, and a clear gap vs the base model in the table above.
- `vi-alpaca` has no held-out split in this pipeline (`docs/stage2_sft.md`), so there is no SFT eval loss; judge by the metrics above and by reading answers.
- Training loss plateaus early (≈2.1 after the first ~0.3 epoch); extra epochs mostly add little.